# Activity counts under node-filtering regimes

**EXPLORATORY** (CLAUDE.md §2: exempt from the TDD gate).

Reads one node-attribution run and asks how many activities each filtering regime keeps, and what those
activities look like.

**Activity** = one SAE feature at one site, `(site_ordinal, feature_idx)`, the default in
`src/discovery/event_log.py`. With `AGGREGATION = "none"` position is part of the identity:
`(site_ordinal, token_idx, feature_idx)`. SAE error nodes (`feature_idx = -1`) are not activities; they are
excluded and only reported for reference.

**Per-example IE** is the unit every regime reads. With `"sum"` it is the signed sum over positions; with
`"none"` it is the row itself (Marks et al., arXiv:2403.19647, App. A.2). `"none"` is only valid when
positions line up across examples.

**Regimes** (add one by adding a function to `REGIMES`):

| Regime | Keeps an activity when | Why it is here |
|---|---|---|
| `marks` (baseline) | \|mean IE\| > `T_N` | Marks et al. §3.1 |
| `recurrence` | \|IE\| > `T_N` in at least `RECURRENCE_FRAC` of examples | Signed means cancel; this keeps effects that are large but inconsistent in sign |
| `mass` | Smallest set holding `MASS` of total \|mean IE\| | Budget set by share of effect instead of an absolute scale, like the node pruning in Ameisen et al. 2025 |
| `ttest_bh` | Mean IE differs from 0 at BH-FDR `FDR_Q` | Evidence instead of magnitude; expected to be permissive at large n |
| `calibrated` | Repo's two-stage screen + recurrence test | The existing HA arm, `src/calibrated_feature_selection.py` |
| `case` | Case by case: an **event** stays if its own \|IE\| > `T_N` | No shared alphabet; each trace keeps only what mattered in it |
| `marks+case` | `marks` alphabet first, then case by case | Removes events of a kept activity that did not pass in that trace |

**Alphabet vs events.** An **event** is one raw row: (example, position, feature). The first five regimes only choose
an alphabet, so a kept activity stays in every trace where it has a non-zero IE, however small that IE is. `case` and
`marks+case` also filter the events, always at single-position level, whatever `AGGREGATION` is. For them, an
activity counts as kept if at least one of its events survives.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                      # output paths are repo-relative
sys.path.insert(0, str(ROOT))
from src.calibrated_feature_selection import calibrated_feature_cols, select_calibrated
from src.selection.threshold import select_by_threshold
from src.utils.config import Config

RUN = Path("outputs/ioi/attributions/nodes/abc_counterfactual")
AGGREGATION = "sum"        # "sum" (non-templatic) | "none" (positions must align)
MAX_EXAMPLES = None        # None = every example; otherwise a sample fixed by SEED
SEED = 42

T_N = 0.1                  # Marks' node threshold; they used 0.1-0.2 on SVA with "none"
RECURRENCE_FRAC = 0.1      # recurrence: min share of examples with |IE| > T_N
MASS = 0.8                 # mass: share of total |mean IE| to keep
FDR_Q = 0.05               # ttest_bh: BH level
INCLUDE_CALIBRATED = True  # slowest regime; tests the raw rows

SYMLOG = 1e-3              # linthresh for the signed-IE axis
PLOT_MAX_POINTS = 2_000_000  # boxplots use at most this many points per regime (sampled with SEED)

## Load the run

In [ ]:
hive = ds.dataset(RUN / "nodes", format="parquet", partitioning="hive")
if not (RUN / "manifest.json").is_file():
    print("WARNING: no manifest.json, so this run is unfinished. Using the examples flushed so far.")

# Site 0 holds a row for every example (error nodes are dense), so it lists them cheaply.
all_ids = np.unique(hive.to_table(columns=["example_idx"], filter=ds.field("site_ordinal") == 0)
                    .column("example_idx").to_numpy())
capped = MAX_EXAMPLES is not None and MAX_EXAMPLES < len(all_ids)
ids = (np.sort(np.random.default_rng(SEED).choice(all_ids, MAX_EXAMPLES, replace=False))
       if capped else all_ids)
N = len(ids)
if N < 2:
    raise ValueError(f"Need at least 2 examples, found {N}.")

# `kind` stays out of the frame: as strings it is most of the memory at n=10k.
raw = (hive.to_table(columns=["example_idx", "site_ordinal", "token_idx", "feature_idx", "attribution"],
                     filter=ds.field("example_idx").isin(ids.tolist()) if capped else None)
       .to_pandas()
       .astype({"example_idx": "int32", "site_ordinal": "int16", "token_idx": "int16",
                "feature_idx": "int32", "attribution": "float32"}))
site_kind = (hive.to_table(columns=["site_ordinal", "kind"], filter=ds.field("example_idx") == int(ids[0]))
             .to_pandas().drop_duplicates().set_index("site_ordinal")["kind"])

if AGGREGATION == "none":
    # Position i only means one thing across examples when lengths and roles match.
    lengths = raw.groupby("example_idx", sort=False)["token_idx"].max()
    if lengths.nunique() > 1:
        raise ValueError(f'AGGREGATION="none" needs one prompt length; found {lengths.nunique()}. '
                         'Filter to one length or use "sum".')
    if (RUN / "examples.parquet").is_file():
        n_templates = pd.read_parquet(RUN / "examples.parquet")["template_key"].nunique()
        if n_templates > 1:
            print(f'WARNING: {n_templates} templates. "none" is valid only if they share position roles '
                  "(Marks' SVA cases do; IOI templates do not).")

print(f"{len(raw):,} rows, {N:,} of {len(all_ids):,} examples, {raw['site_ordinal'].nunique()} sites")

## Per-example IE and per-activity statistics

Activities are packed into one integer key so grouping stays affordable at ~10⁸ rows. An activity absent from
an example had zero effect there, so every mean divides by all `N` examples, as `src/selection/aggregate.py`
does.

In [ ]:
BY_POSITION = AGGREGATION == "none"


def activity_key(df):
    # 40+ bits site | 16 bits token+1 (0 when positions are summed) | 24 bits feature+1
    token = df["token_idx"].to_numpy(np.int64) + 1 if BY_POSITION else 0
    return (df["site_ordinal"].to_numpy(np.int64) << 40) | (token << 24) | (df["feature_idx"].to_numpy(np.int64) + 1)


raw_key = activity_key(raw)
per_example = pd.DataFrame({"key": raw_key, "example_idx": raw["example_idx"].to_numpy(),
                            "ie": raw["attribution"].to_numpy(np.float64)})
if AGGREGATION == "sum":
    per_example = per_example.groupby(["key", "example_idx"], sort=False)["ie"].sum().reset_index()

table = (per_example.assign(ie2=per_example["ie"] ** 2, hits=per_example["ie"].abs() > T_N)
         .groupby("key", sort=False)[["ie", "ie2", "hits"]].sum())
del per_example
table["ie"] /= N                                      # mean over ALL examples
var = ((table["ie2"] - N * table["ie"] ** 2) / (N - 1)).clip(lower=0)
with np.errstate(divide="ignore", invalid="ignore"):
    t_stat = table["ie"].to_numpy() / np.sqrt(var.to_numpy() / N)
table["p"] = np.nan_to_num(2 * stats.t.sf(np.abs(t_stat), N - 1), nan=1.0)

key = table.index.to_numpy()
table["site_ordinal"] = key >> 40
table["token_idx"] = ((key >> 24) & 0xFFFF) - 1       # -1 when positions are summed, as in aggregate()
table["feature_idx"] = (key & 0xFFFFFF) - 1
table["kind"] = pd.Categorical(table["site_ordinal"].map(site_kind))

errors = table[table["feature_idx"] == -1]
features = table[table["feature_idx"] != -1]
print(f"{len(features):,} activities, {len(errors):,} error nodes")

## Regimes

In [ ]:
def marks(a):
    # Marks et al., arXiv:2403.19647, §3.1: |mean IE| > T_N, via the pipeline's own selector.
    return select_by_threshold(a, T_N)["selected"].to_numpy()


def recurrence(a):
    return (a["hits"] / N >= RECURRENCE_FRAC).to_numpy()


def mass(a):
    # Share-of-effect budget, like node pruning in Ameisen et al. 2025 (Circuit Tracing).
    abs_ie = a["ie"].abs().to_numpy()
    order = np.argsort(-abs_ie)
    n_keep = np.searchsorted(np.cumsum(abs_ie[order]) / abs_ie.sum(), MASS) + 1
    keep = np.zeros(len(a), dtype=bool)
    keep[order[:n_keep]] = True
    return keep


def ttest_bh(a):
    return stats.false_discovery_control(a["p"].to_numpy(), method="bh") <= FDR_Q


def calibrated(a):
    cfg = Config()
    cfg.selection.aggregation = AGGREGATION
    feats = select_calibrated(raw.assign(is_error=raw["feature_idx"] == -1), cfg)
    kept = feats.loc[feats["selected"], calibrated_feature_cols(cfg)]
    return a.index.isin(activity_key(kept.assign(token_idx=kept.get("token_idx", 0))))


REGIMES = {"marks": marks, "recurrence": recurrence, "mass": mass, "ttest_bh": ttest_bh}
if INCLUDE_CALIBRATED:
    REGIMES["calibrated"] = calibrated

selected = pd.DataFrame({name: fn(features) for name, fn in REGIMES.items()}, index=features.index)
print(f"Error nodes over T_N, for reference (not activities): {(errors['ie'].abs() > T_N).sum()} of {len(errors)}")

## Events and counts

`activities` is the size of the regime's alphabet. Per example, **activities** counts distinct kept activities and
**events** counts surviving rows; they differ under `"sum"`, where one activity can fire at several positions.
Events with IE exactly 0 (features active but unchanged between clean and patch) never count.

In [ ]:
# regime -> the alphabet it starts from (None = every activity)
CASE_REGIMES = {"case": None, "marks+case": "marks"}

# From here on, only feature rows matter: `rows` is each event's row in `features`.
row = features.index.get_indexer(raw_key)
on_feature = row >= 0
rows, event_ie = row[on_feature], raw["attribution"].to_numpy(np.float64)[on_feature]
example_pos = np.searchsorted(ids, raw["example_idx"].to_numpy()[on_feature])
del raw, raw_key, row, on_feature

moved = event_ie != 0
rng = np.random.default_rng(SEED)


def sample(values):
    if len(values) <= PLOT_MAX_POINTS:
        return values
    return values[rng.integers(0, len(values), PLOT_MAX_POINTS)]


def trace_counts(mask):
    "(events per example, distinct activities per example, sampled event IEs) for the events in `mask`."
    events = np.bincount(example_pos[mask], minlength=N)
    if BY_POSITION:                     # the activity includes the position, so events are distinct
        return events, events, sample(event_ie[mask])
    pairs = np.unique(rows[mask] * N + example_pos[mask])
    return events, np.bincount(pairs % N, minlength=N), sample(event_ie[mask])


traces = {"all": trace_counts(moved)}
for name in list(selected):             # alphabet regimes: every moved event of a kept activity
    traces[name] = trace_counts(moved & selected[name].to_numpy()[rows])

for name, base in CASE_REGIMES.items():
    mask = np.abs(event_ie) > T_N
    if base is not None:
        mask &= selected[base].to_numpy()[rows]
    alphabet = np.zeros(len(features), dtype=bool)
    alphabet[rows[mask]] = True         # kept if at least one event survives
    selected[name] = alphabet
    traces[name] = trace_counts(mask)

index = pd.Index(ids, name="example_idx")
per_example_events = pd.DataFrame({k: v[0] for k, v in traces.items()}, index=index)
per_example_counts = pd.DataFrame({k: v[1] for k, v in traces.items()}, index=index)
regimes = list(selected)

summary = pd.DataFrame({
    "activities": selected.sum(),
    "% of activities": (100 * selected.mean()).round(3),
    "events": per_example_events[regimes].sum(),
    "activities/example (median)": per_example_counts[regimes].median(),
    "activities/example (max)": per_example_counts[regimes].max(),
    "events/example (median)": per_example_events[regimes].median(),
    "events/example (max)": per_example_events[regimes].max(),
})
display(summary)
display(pd.DataFrame({name: features.loc[selected[name], "kind"].value_counts() for name in selected})
        .fillna(0).astype(int).rename_axis("kind"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, column, total in zip(axes, ["activities", "events"], [len(features), int(per_example_events["all"].sum())]):
    bars = ax.bar(summary.index, summary[column].clip(lower=0.5), color="#4C72B0")
    ax.bar_label(bars, labels=[f"{c:,}" for c in summary[column]], fontsize=8)
    ax.axhline(total, color="grey", ls="--", lw=1, label=f"before filtering ({total:,})")
    ax.set_yscale("log")
    ax.set_ylabel(f"{column} kept (log)")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(frameon=False)
fig.suptitle(f"Kept per regime ({AGGREGATION}, N={N:,}, T_N={T_N})")
plt.tight_layout()
plt.show()

## Distributions

`all` is everything before filtering. Quartiles are computed on at most `PLOT_MAX_POINTS` points per regime.

- **Activity-level IE:** one point per kept activity, its mean IE over all examples.
- **Event-level IE:** one point per surviving event, its own IE. This shows what a mean filter leaves in the traces.
- **Counts:** one point per example.

In [ ]:
def boxes(ax, data, ylabel, title):
    ax.boxplot([sample(v) for v in data.values()], flierprops={"markersize": 1.5, "alpha": 0.4, "rasterized": True},
               medianprops={"color": "#C44E52"})
    ax.set_xticks(range(1, len(data) + 1), list(data), rotation=25)
    ax.set_ylabel(ylabel)
    ax.set_title(title)


def ie_panels(data, level):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    boxes(axes[0], {k: np.abs(v[v != 0]) for k, v in data.items()}, f"|{level} IE| (log)",
          f"|IE| per {level} (exact zeros omitted)")
    axes[0].set_yscale("log")
    boxes(axes[1], data, f"{level} IE (symlog)", f"Signed IE per {level}")
    axes[1].set_yscale("symlog", linthresh=SYMLOG)
    axes[1].axhline(0, color="grey", lw=0.8)
    plt.tight_layout()
    plt.show()


mean_ie = features["ie"].to_numpy()
ie_panels({"all": mean_ie, **{name: mean_ie[selected[name].to_numpy()] for name in regimes}}, "activity")
ie_panels({name: traces[name][2] for name in ["all", *regimes]}, "event")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, frame, label in zip(axes, [per_example_counts, per_example_events], ["activities", "events"]):
    boxes(ax, {name: frame[name].to_numpy() for name in frame}, f"{label} per example (symlog)",
          f"Kept {label} per example")
    ax.set_yscale("symlog", linthresh=1)
plt.tight_layout()
plt.show()